# Rotax 914 ULF - Step 3: How bad is it?

**In one sentence:** For each fault that is present, estimates how far it has progressed - from 0 (just started) to 1 (fully developed).

| Goes in | Comes out |
|---|---|
| The same 128-second window | One severity per fault type, 0 to 1 |

Starts from what step 2 (*Which part is failing?*) already learned.

Run the cells top to bottom with **Shift+Enter**. If this step was already trained with `run.py`, training just loads the saved model in a second; otherwise it trains here.

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '914', 'severity'
setup(ENGINE, PHASE)

### 1. How far faults have progressed in the data
A fault starts small and grows over many flights. This is how far along the
faults in the training data are when the model sees them.

In [ ]:
sev = load_rows(ENGINE, P.FM_COLS, split="train")[P.FM_COLS].max(axis=1)
sev = sev[sev >= P.DETECTION_THRESHOLD]
plt.figure(figsize=(8, 3.2)); plt.hist(sev, 40, color=FAULTY, alpha=0.8)
plt.xlabel("fault severity (0 = just started, 1 = fully developed)"); plt.ylabel("rows")
plt.title("How far faults have progressed"); plt.tight_layout(); plt.show()

### 2. Train the model (or load it)

In [ ]:
payload = train_or_load(ENGINE, PHASE)
print(f"Done. Trained for {payload.get('epochs_run', '?')} passes over the data.")

### 3. Scorecard
Measured on the test flights. The quality bar catches a model that did not learn - it is a safety check, not a target.

In [ ]:
scorecard(ENGINE, PHASE)

### 4. See it working on flights it never saw
Each dot is one window with a fault: its true severity against the model's
estimate, for the main fault in that window. Dots on the dashed line are exact.

In [ ]:
r = predict_test(ENGINE, PHASE)
i = r["true"].argmax(1)
true = r["true"][np.arange(len(i)), i]
pred = r["pred"][np.arange(len(i)), i]
m = true >= P.DETECTION_THRESHOLD
plt.figure(figsize=(5, 5)); plt.scatter(true[m], pred[m], s=3, alpha=0.3, color=FAULTY)
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlim(0, 1); plt.ylim(0, 1)
plt.xlabel("true severity"); plt.ylabel("predicted severity")
plt.title("Each dot is one faulty window"); plt.tight_layout(); plt.show()

### 5. What this means

In [ ]:
err = np.abs(pred[m] - true[m])
print(f"On windows with a fault, severity is off by {err.mean():.2f} on average "
      f"(median {np.median(err):.2f}) on the 0-1 scale.")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |